# Du CSV aux valeurs réellement calculées

**Exemples CSV pour la séance du 25 septembre.**

**Parcours projeté : sections 1 à 4, puis 6 et 10.** Les sections
5 et 7 à 9, les passages « référence facultative » et l'annexe sont
disponibles pour approfondir ensuite. Les premières notions sont la
lecture CSV, les types, les valeurs manquantes et `lambda`. Le cours
compare aussi NumPy et pandas. Ce notebook explique les fonctions utilisées pour lire et transformer
les données. Les mêmes règles de lecture sont utilisées dans
[`data_io.py`](../sql/data_io.py), avec les clients et leurs commandes.

Nous gardons les trois colonnes des tickets : `ticket_id`, `channel`,
`resolution_minutes`. Ce petit CSV sert à voir chaque transformation.
Une cellule vide, le texte `NA`, le texte `unknown` et le nombre zéro
n'ont pas spontanément le même sens.

Pour chaque exemple : **prédire → exécuter → modifier → expliquer**.
Lisez la question avant la cellule. Travaillez dans VS Code/Jupyter,
avec l'environnement Python du cours. Exécutez les cellules dans
l'ordre ; un redémarrage du noyau efface les variables en mémoire.
Ce notebook n'écrit aucun fichier.

## 1. Un CSV est d'abord du texte

**Prédire :** quels éléments doivent rester du texte ? Les zéros de
`001` sont-ils une quantité ? `StringIO` permet de lire ce texte comme
un fichier ; avec un vrai CSV, on donne son chemin à `read_csv`.

In [ ]:
from io import StringIO

import numpy as np
import pandas as pd
from IPython.display import display

csv_text = """ticket_id,channel,resolution_minutes
001,phone,30
002,chat,
003,email,NA
004,phone,0
005,email,unknown
"""
print(csv_text)
print("pandas:", pd.__version__, "NumPy:", np.__version__)

## 2. Laisser pandas choisir les types

**Prédire :** quelle colonne sera numérique ? Quelles cellules seront
reconnues comme manquantes ? Inspectez ensuite le tableau **et** les
types. L'affichage `30` ne prouve pas que la valeur est un nombre.

In [ ]:
inferred = pd.read_csv(StringIO(csv_text))
display(inferred)
display(inferred.dtypes)
print("Missing durations:", int(inferred["resolution_minutes"].isna().sum()))

Dans cet exemple, ce choix automatique peut faire perdre les zéros des identifiants.
Le texte `unknown` empêche une colonne entièrement numérique.
Avec pandas 3, les colonnes reconnues automatiquement comme du texte portent souvent le
dtype `str` ; pandas 2 utilisait généralement `object`.

## 3. Conserver les cellules d'origine

`dtype="string"` demande un type de texte qui accepte aussi les valeurs manquantes. Avec
`keep_default_na=False`, les chaînes `""` et `"NA"` restent du texte.
**Prédire :** combien de valeurs `isna()` trouvera-t-il maintenant ?

In [ ]:
raw = pd.read_csv(StringIO(csv_text), dtype="string", keep_default_na=False)
display(raw)
display(raw.dtypes)
print("Missing durations:", int(raw["resolution_minutes"].isna().sum()))

**Modifier :** remplacez temporairement `unknown` par `120` dans le CSV,
puis rejouez les lectures. Expliquez ce que le choix de lecture change.
Rétablissez le CSV initial avant la suite.

Un **type Python** décrit un objet ; un **dtype pandas** décrit le
stockage et le comportement d'une colonne. Une `Series` est un objet
Python, même lorsque ses éléments sont numériques.

In [ ]:
durations_text = raw["resolution_minutes"]
print(type(raw).__name__, type(durations_text).__name__)
print("Column dtype:", durations_text.dtype)
print("First value type:", type(durations_text.iloc[0]).__name__)
print("Exact cell contents:", durations_text.map(repr).tolist())

## 4. Déclarer une règle de lecture par colonne

**Hypothèse limitée à cet exercice :** dans `resolution_minutes`,
`""`, `"NA"` et `"unknown"` signifient « durée non renseignée ».
Ces textes ne deviennent pas manquants dans les autres colonnes.
`Float64`, avec une majuscule, accepte nombres et valeurs manquantes.

Conservez aussi `raw` : le tableau numérique ne distingue plus ces
trois textes d'origine. Cette règle de lecture est propre à notre exemple,
pas une règle universelle de nettoyage.

In [ ]:
typed = pd.read_csv(
    StringIO(csv_text),
    dtype={"ticket_id": "string", "channel": "string",
           "resolution_minutes": "Float64"},
    keep_default_na=False,
    na_values={"resolution_minutes": ["", "NA", "unknown"]},
)
display(typed)
display(typed.dtypes)

## 5. Convertir après avoir inspecté

Une autre démarche conserve d'abord le texte puis le convertit.
`errors="raise"` arrête la conversion devant un texte non convertible.
Ici l'erreur est capturée pour pouvoir poursuivre le notebook.
**Prédire :** quel texte provoquera le message ?

In [ ]:
duration_text = raw["resolution_minutes"].str.strip()
try:
    strict_durations = pd.to_numeric(duration_text, errors="raise")
except ValueError as error:
    print(type(error).__name__, str(error))

`errors="coerce"` produit des valeurs manquantes quand la conversion
échoue. Cela ne fournit pas la justification d'une exclusion.
**Inspecter :** rapprochez chaque résultat du texte initial.
**Modifier :** essayez `"erreur"` au lieu de `"unknown"` : la conversion
ne permet pas, à elle seule, de dire ce que ce texte signifie.

In [ ]:
numeric_durations = pd.to_numeric(duration_text, errors="coerce")
conversion_audit = pd.DataFrame({
    "original": duration_text,
    "numeric": numeric_durations,
    "is_missing": numeric_durations.isna(),
    "is_empty_text": duration_text == "",
})
display(conversion_audit)

## 6. Repérer les valeurs manquantes

**Référence facultative — détail des marqueurs.** Le tableau suivant
se consulte sans apprendre tous les noms. Reprenez ensuite le socle
avec `isna()` : une même méthode répond au besoin pour ces types.

Le marqueur dépend du dtype : `NaN` pour `float64`, `<NA>` pour les
types qui acceptent les valeurs manquantes ci-dessous, `NaT` pour les dates. Le dtype `str` de
pandas 3 utilise `NaN`, contrairement à `string` qui utilise `<NA>`.
**Question :** faut-il mémoriser une comparaison différente pour
chaque marqueur ? Vérifiez la dernière colonne.

In [ ]:
examples = {
    "float64": pd.Series([0.0, np.nan], dtype="float64"),
    "Float64": pd.Series([0.0, pd.NA], dtype="Float64"),
    "Int64": pd.Series([0, pd.NA], dtype="Int64"),
    "boolean": pd.Series([False, pd.NA], dtype="boolean"),
    "string": pd.Series(["phone", pd.NA], dtype="string"),
    "datetime": pd.Series([pd.Timestamp("2026-09-22"), pd.NaT]),
}
if int(pd.__version__.split(".")[0]) >= 3:
    examples["str"] = pd.Series(["phone", np.nan], dtype="str")
rows = [{"dtype": str(values.dtype), "missing_marker": repr(values.iloc[1]),
         "isna": values.isna().tolist()} for values in examples.values()]
display(pd.DataFrame(rows))

Utilisez **`isna()` / `notna()`**, pas `== np.nan` ni `== pd.NA`.
**Prédire :** les trois tests ci-dessous répondent-ils pareil ?
Une comparaison avec une valeur inconnue peut rester inconnue.

In [ ]:
print("NaN equals NaN:", np.nan == np.nan)
print("NA equals NA:", pd.NA == pd.NA)
print("Missing markers:", pd.isna([np.nan, pd.NA, pd.NaT]).tolist())
display(typed["resolution_minutes"].isna())

**Zéro et `False` sont des valeurs connues.** Remplacer une durée
inconnue par zéro inventerait une résolution immédiate. De même,
`False` et « réponse inconnue » décrivent deux situations différentes.
`bool(pd.NA)` lève une erreur : le manque n'est pas implicitement faux.

In [ ]:
numeric_example = pd.Series([0, pd.NA, 30], dtype="Float64")
boolean_example = pd.Series([False, pd.NA, True], dtype="boolean")
display(pd.DataFrame({"duration": numeric_example,
                      "missing_duration": numeric_example.isna(),
                      "answer": boolean_example,
                      "missing_answer": boolean_example.isna()}))

## 7. Quelles lignes compter pour calculer une proportion ?

`len` compte les lignes ; `count` compte les valeurs non manquantes.
**Prédire à la main :** parmi les durées connues du petit CSV, quelle
part atteint 30 minutes ? Le zéro reste une durée connue.
Annoncez toujours combien de durées ne sont pas renseignées.

In [ ]:
durations = typed["resolution_minutes"]
valid_durations = durations.dropna()
numerator = int((valid_durations >= 30).sum())
denominator = len(valid_durations)
print("Rows:", len(durations), "Known durations:", durations.count())
print("Missing:", int(durations.isna().sum()))
print("Numerator:", numerator, "Denominator:", denominator)
print("Frequency among known durations:", numerator / denominator)

## 8. Une sélection lisible, une copie explicite

Un masque contient une décision par ligne. Ici, seules les durées
renseignées et positives ou nulles entrent dans le calcul.
`fillna(False)` concerne **le masque**, pas les durées.
**Prédire :** quels identifiants restera-t-il ?

In [ ]:
prepared = raw.copy()
prepared["resolution_minutes"] = numeric_durations
has_duration = prepared["resolution_minutes"].notna()
is_nonnegative = prepared["resolution_minutes"] >= 0
keep = (has_duration & is_nonnegative).fillna(False)
valid_tickets = prepared.loc[keep].copy()
display(valid_tickets)

**Référence facultative — normaliser puis sélectionner.**

Les méthodes `.str` traitent chaque texte d'une Series. Une méthode
Python comme `.strip()` traite un seul texte. La normalisation d'un
canal repose ici sur la règle connue « espaces extérieurs et casse
sans signification ». Un canal inconnu demande une vérification.

In [ ]:
channel_variants = pd.Series([" Phone ", "EMAIL", "chat", pd.NA], dtype="string")
normalized_channels = channel_variants.str.strip().str.lower()
known_channels = normalized_channels.isin(["phone", "chat", "email"])
display(pd.DataFrame({"original": channel_variants,
                      "normalized": normalized_channels,
                      "known": known_channels}))
print("One Python string:", " Phone ".strip().lower())

**À modifier puis expliquer :** ajoutez le canal `"web"` aux variantes.
Le code corrige-t-il son sens ? Dans la cellule suivante, remplacez
`phone` par `email` et expliquez le résultat vide.

In [ ]:
selected_channel = "phone"
in_channel = valid_tickets["channel"] == selected_channel
selected_tickets = valid_tickets.loc[in_channel]
display(selected_tickets)
print("Known durations:", selected_tickets["resolution_minutes"].count())

## 9. NumPy et pandas sur les mêmes valeurs

Un tableau NumPy porte ici des valeurs numériques repérées par
position. Une Series pandas ajoute des étiquettes d'index.
Une DataFrame organise plusieurs colonnes. pandas et NumPy se
complètent ; nous utilisons pandas pour les tickets.

**Prédire :** mêmes nombres, mêmes moyenne et écart-type ? Nous
imposons `ddof=1` aux deux calculs : leurs valeurs par défaut diffèrent.

In [ ]:
duration_array = np.array([30.0, 60.0, 90.0])
duration_series = pd.Series(duration_array, index=["001", "002", "003"])
print("NumPy:", duration_array, "dtype:", duration_array.dtype)
display(duration_series)
print("Means:", duration_array.mean(), duration_series.mean())
print("Sample standard deviations:", duration_array.std(ddof=1),
      duration_series.std(ddof=1))
print("Ticket 002:", duration_series.loc["002"])

## Point de passage — modifier sans deviner

**Avant d'exécuter :** une durée de plus devient connue ci-dessous.
Prédisez le nombre de lignes, le nombre de durées connues et la
fréquence à partir de 30 minutes. Expliquez votre dénominateur.
Puis remplacez `120` par `0` et expliquez ce qui change.

In [ ]:
modified_csv = csv_text.replace("005,email,unknown", "005,email,120")
modified = pd.read_csv(
    StringIO(modified_csv),
    dtype={"ticket_id": "string", "channel": "string",
           "resolution_minutes": "Float64"},
    keep_default_na=False,
    na_values={"resolution_minutes": ["", "NA", "unknown"]},
)
known = modified["resolution_minutes"].dropna()
print("Rows:", len(modified), "Known:", len(known))
print("Frequency:", int((known >= 30).sum()) / len(known))

## 10. Fonction nommée, lambda et appelant

Une `lambda` est une fonction limitée à une expression. `map`
l'appelle pour chaque élément de l'itérable ; `list` matérialise
les résultats. **Prédire :** les deux calculs donnent-ils les mêmes
valeurs ? Quel objet `map` renvoie-t-il avant `list` ?

In [ ]:
def minutes_to_hours(minutes):
    return minutes / 60


examples = [30, 60, 90]
named_hours = list(map(minutes_to_hours, examples))
lambda_hours = list(map(lambda minutes: minutes / 60, examples))
print(named_hours, lambda_hours)
assert named_hours == lambda_hours == [0.5, 1.0, 1.5]
print("map returns:", type(map(minutes_to_hours, examples)).__name__)

`Series.map` appelle la fonction sur les valeurs de la colonne.
`na_action="ignore"` épargne ici les valeurs manquantes à la lambda.
Pour une simple division numérique, l'opération vectorisée est plus
directe. **Comparer :** résultats, dtypes et lisibilité.

In [ ]:
durations = typed["resolution_minutes"]
mapped_hours = durations.map(lambda minutes: minutes / 60,
                             na_action="ignore")
direct_hours = durations / 60
display(pd.DataFrame({"minutes": durations,
                      "mapped": mapped_hours,
                      "direct": direct_hours}))
pd.testing.assert_series_equal(mapped_hours.astype("Float64"),
                               direct_hours.astype("Float64"))

`sorted(key=...)` transmet chaque élément à une fonction de clé.
Ici la clé est d'abord la longueur du nom, puis le nom lui-même.
Une lambda convient parce que cette clé tient en une expression ;
un traitement à plusieurs étapes mérite un `def` nommé.

In [ ]:
names = ["phone", "email", "chat"]
by_length = sorted(names, key=lambda name: (len(name), name))
print(by_length)
assert by_length == ["chat", "email", "phone"]

## Annexe facultative — position et étiquette

Les étiquettes comptent dans les opérations pandas. Les tableaux
NumPy ci-dessous additionnent par position. Les Series additionnent
après alignement sur les identifiants. **Prédire** avant d'exécuter.

In [ ]:
first = pd.Series([10, 20], index=["001", "002"])
second = pd.Series([100, 200], index=["002", "001"])
print("NumPy positional sum:", first.to_numpy() + second.to_numpy())
print("pandas labelled sum:")
display(first + second)

## Repères officiels

- [pandas — lecture CSV et paramètres](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [pandas — conversion numérique](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)
- [pandas — valeurs manquantes](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas 3 — types textuels](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html)
- [pandas — Series.count](https://pandas.pydata.org/docs/reference/api/pandas.Series.count.html)
- [NumPy — écart-type et ddof](https://numpy.org/doc/stable/reference/generated/numpy.std.html)
- [pandas — alignement des Series](https://pandas.pydata.org/docs/user_guide/dsintro.html#series)

**À savoir refaire :** lire sans perdre l'identifiant, distinguer texte
et nombre, détecter le manque, sélectionner des valeurs valides,
expliquer un dénominateur et reconnaître qui appelle une lambda.